# Лабораторная 07. shuffle partitions и AQE

Цель: увидеть, как `spark.sql.shuffle.partitions` и AQE влияют на количество post-shuffle tasks.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-07-aqe').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
print('Spark UI:', spark.sparkContext.uiWebUrl)

## Часть 1. AQE выключен, shuffle partitions = 200
После action посмотрите stage после shuffle. Ожидайте около 200 reduce tasks.

In [ ]:
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '200')
q_200 = orders.groupBy('customer_id').agg(F.count('*').alias('orders_cnt'))
q_200.explain('formatted')
q_200.count()

## Часть 2. AQE выключен, shuffle partitions = 8

In [ ]:
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '8')
q_8 = orders.groupBy('customer_id').agg(F.count('*').alias('orders_cnt'))
q_8.explain('formatted')
q_8.count()

## Часть 3. AQE включен, initial shuffle partitions = 200
AQE может объединить маленькие post-shuffle partitions. Смотрите SQL tab: adaptive plan и фактическое количество tasks.

In [ ]:
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.shuffle.partitions', '200')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.adaptive.advisoryPartitionSizeInBytes', '16m')
q_aqe = orders.groupBy('customer_id').agg(F.count('*').alias('orders_cnt'))
q_aqe.explain('formatted')
q_aqe.count()
q_aqe.explain('formatted')

Заполните:

| Часть | AQE | `spark.sql.shuffle.partitions` | Tasks после shuffle в Spark UI | Что изменилось |
|---|---|---:|---:|---|
| 1 | false | 200 | ? | ? |
| 2 | false | 8 | ? | ? |
| 3 | true | 200 | ? | ? |

Вопросы:

- Сколько partitions было задано изначально?
- Сколько реально получилось?
- Почему AQE может уменьшить количество маленьких tasks?